In [1]:
import numpy as np

In [2]:
np.set_printoptions(precision=5)
np.set_printoptions(suppress=True)
np.set_printoptions(linewidth=np.inf)
np.set_printoptions(threshold=np.inf)

In [3]:
def sigma_ham(sites: int, sigma: np.ndarray, i: int) -> np.ndarray:

    states = 2

    K_H = np.zeros((states**sites, states**sites), dtype=complex)

    n_lambda = states ** (i)
    n_mu = states ** (sites - i - 1)

    for p in range(states):
        for p_prime in range(states):

            val = sigma[p, p_prime]

            if val == 0:
                continue

            for Lambda in range(int(n_lambda)):
                for mu in range(int(n_mu)):

                    i = mu + p * n_mu + Lambda * states * n_mu
                    j = mu + p_prime * n_mu + Lambda * states * n_mu

                    K_H[i, j] = val
    return K_H

In [4]:
def sibma_kron(site: int, sigma: np.ndarray, i: int):

    I_f = np.eye(2**(i))
    I_e = np.eye(2**(site - i - 1))

    single_term = np.kron(I_f, np.kron(sigma, I_e))

    return single_term

In [ ]:
def heis_ham_kron(site: int, J_x: int, J_y: int, J_z: int, h_x: int, h_y: int, h_z: int):

    sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
    sigma_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
    sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)

    single_term = np.zeros((2**site, 2**site), dtype=complex)
    double_term = np.zeros((2**site, 2**site), dtype=complex)

    for i in range(site):

        if h_x != 0:
            single_term += h_x * sibma_kron(site, sigma_x, i)

        if h_y != 0:
            single_term += h_y * sibma_kron(site, sigma_y, i)

        if h_z != 0:
            single_term += h_z * sibma_kron(site, sigma_z, i)

    for i in range(site-1):

        if J_x != 0:
            x_term =  sibma_kron(site, sigma_x, i)
            x_term_int =  sibma_kron(site, sigma_x, i+1)

            double_term += J_x * x_term @ x_term_int

        if J_y != 0:
            y_term =  sibma_kron(site, sigma_y, i)
            y_term_int =  sibma_kron(site, sigma_y, i+1)

            double_term += J_y * y_term @ y_term_int

        if J_z != 0:
            z_term =  sibma_kron(site, sigma_z, i)
            z_term_int =  sibma_kron(site, sigma_z, i+1)

            double_term += J_z * z_term @ z_term_int

    return single_term, double_term

In [ ]:
def heis_ham(site: int, J_x: int, J_y: int, J_z: int, h_x: int, h_y: int, h_z: int):

    sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
    sigma_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
    sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)

    single_term = np.zeros((2**site, 2**site), dtype=complex)
    double_term = np.zeros((2**site, 2**site), dtype=complex)

    for i in range(site):

        if h_x != 0:
            single_term += h_x * sigma_ham(site, sigma_x, i)

        if h_y != 0:
            single_term += h_y * sigma_ham(site, sigma_y, i)

        if h_z != 0:
            single_term += h_z * sigma_ham(site, sigma_z, i)

    for i in range(site-1):

        if J_x != 0:
            x_term =  sigma_ham(site, sigma_x, i)
            x_term_int =  sigma_ham(site, sigma_x, i+1)

            double_term += J_x * x_term @ x_term_int

        if J_y != 0:
            y_term =  sigma_ham(site, sigma_y, i)
            y_term_int =  sigma_ham(site, sigma_y, i+1)

            double_term += J_y * y_term @ y_term_int

        if J_z != 0:
            z_term =  sigma_ham(site, sigma_z, i)
            z_term_int =  sigma_ham(site, sigma_z, i+1)

            double_term += J_z * z_term @ z_term_int

    return single_term,  double_term

In [11]:
state = 2
site = 11

g = 1

In [8]:
sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
sigma_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)

i = 3

In [9]:
s = sigma_z

In [10]:
single_term = sigma_ham(site, s, 0)
single_term_int =  sigma_ham(site, s, 1)

single_term_kron = sibma_kron(site, s, 0)
single_term_int_kron = sibma_kron(site, s, 1)

In [11]:
single_term_kron.shape

(16, 16)

In [12]:
double_term = single_term @ single_term_int
double_term_kron = single_term_kron @ single_term_int_kron

In [13]:
print("Single:", np.array_equal(single_term, single_term_kron))
print("Single_int:", np.array_equal(single_term_int, single_term_int_kron))
print("Double:", np.array_equal(double_term, double_term_kron))

Single: True
Single_int: True
Double: True


In [12]:
H_heis_kron = heis_ham_kron(site, 1, 1, 1, 0, 0, 1)

In [13]:
H_heis = heis_ham(site, 1, 1, 1, 0, 0, 1)

In [14]:
np.array_equal(H_heis_kron, H_heis)

True